In [0]:
%sql
 CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
 import requests

 # Define the current catalog
 catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

 # Define the base path using the current catalog
 volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

 # List of files to download
 files = ["2019.csv", "2020.csv", "2021.csv"]

 # Download each file
 for file in files:
     url = f"https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/{file}"
     response = requests.get(url)
     response.raise_for_status()

     # Write to Unity Catalog volume
     with open(f"{volume_base}/{file}", "wb") as f:
         f.write(response.content)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orderSchema = StructType([
     StructField("SalesOrderNumber", StringType()),
     StructField("SalesOrderLineNumber", IntegerType()),
     StructField("OrderDate", DateType()),
     StructField("CustomerName", StringType()),
     StructField("Email", StringType()),
     StructField("Item", StringType()),
     StructField("Quantity", IntegerType()),
     StructField("UnitPrice", FloatType()),
     StructField("Tax", FloatType())
])
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', format='csv', schema=orderSchema)
display(df.limit(100))

In [0]:
from pyspark.sql.functions import col, round, format_number
df = df.dropDuplicates()
# 如果转成float类型，数据即便之前round了，还是会显示多位
# 所以要先转float，再round才有效
df = df.withColumn('Tax', col('Tax').cast("float"))
df = df.withColumn('Tax', round(col('UnitPrice') * 0.08,2))
# df = df.withColumn(
#     'Tax',
#     (col('UnitPrice') * 0.08).cast(DecimalType(10, 2))
# )
#df = df.withColumn('Tax', format_number(col('UnitPrice') * 0.08, 2))
display(df.limit(100))

In [0]:
# customers = df['CustomerName', 'Email']
customers = df.select("CustomerName", "Email")
print(customers.count())
print(customers.distinct().count())
display(customers.distinct())

In [0]:
productSales = df.select("Item", "Quantity").groupBy("Item").sum().withColumnRenamed("sum(Quantity)", "totalQty")
display(productSales)

In [0]:
yearlySales = df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

In [0]:
# create a temporary view that can then be used directly with SQL statements.
df.createOrReplaceTempView("salesorders")

In [0]:
%sql
    
SELECT YEAR(OrderDate) AS OrderYear,
       SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue
FROM salesorders
GROUP BY YEAR(OrderDate)
ORDER BY OrderYear;